# ToolStrategy详解
ToolStrategy通过 工具调用 （Tool Calling/Function Calling）实现结构化输出，所以LangChain会在消息列表末尾 追加一条ToolMessage ，让整个链路完整。但实际上没有实际的工具执行，这是一条伪消息。

ToolStrategy的配置包含三个主要参数：
```python
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[bool, str, type[Exception], tuple[type[Exception],
...], Callable[[Exception], str]]
```

- schema（必需参数）：与提供商策略的schema参数功能一致，支持 `Pydantic模型` 、`TypedDict` 、 `JSON Schema` 、 `数据类(@dataclass)` ，同时还支持联合类型 `Union[类型1, 类型2]` （允许模型根据输入内容选择最匹配的数据结构）。
  
- tool_message_content（可选参数）：用于自定义生成结构化输出时，会话历史中记录的提示信息。默认使用展示输出数据的标准响应语句。
  
- handle_errors（可选参数）：用于指定数据校验失败时的重试策略， 默认值为True 。

In [19]:
from langchain.agents import create_agent
from dotenv import load_dotenv
import os
from langchain_qwq import ChatQwen
from rich import print as rprint
from langchain.tools import tool

load_dotenv(override=True)
model = ChatQwen(
    model="qwen3.6-27b",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

##  结构化输出：schema参数

### 输出模式：Pydantic类型
Pydantic类型的Schema支持数据验证，是优先推荐使用的方式。

In [4]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
    ]
})

rprint(response)
for msg in response["messages"]:
    msg.pretty_print()
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='11deb1f9-1465-4b33-b443-245adb7ce148'
        ),
        AIMessage(
            content='好的，我将为您从这段话中抽取联系信息',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 
'用户要求从文本中抽取结构化信息，包括姓名、邮箱和手机号。文本中提到："小明的邮箱地址为：songhk@atguigu.com，手机号
：12345678912"\n\n所以：\n- 姓名：小明\n- 邮箱：songhk@atguigu.com\n- 
手机号：12345678912\n\n我需要调用ContactInfo工具来记录这些信息。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 178,
                    'prompt_tokens': 344,
                    'total_tokens': 522,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 82,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'dashscope',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-30bdac02-0886-925f-9e86-297a1adf3c8d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f1dd2-9808-7d13-b0ff-5bfd2099c340-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_9cf566830e8f45e09bb9247b',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 344,
                'output_tokens': 178,
                'total_tokens': 522,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 82}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='72301f2b-e7c6-472b-b74c-e93fb7fffe0c',
            tool_call_id='call_9cf566830e8f45e09bb9247b'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912
================================== Ai Message ==================================

好的，我将为您从这段话中抽取联系信息
Tool Calls:
  ContactInfo (call_9cf566830e8f45e09bb9247b)
 Call ID: call_9cf566830e8f45e09bb9247b
  Args:
    name: 小明
    email: songhk@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'


In [3]:
# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"



In [4]:
@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"
    
    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

In [ ]:
from typing import Literal

# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] =Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None,description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

In [6]:
from langchain_core.messages import SystemMessage
# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
    "请分析指定客户的情况："
    "1. 先搜索客户数据库了解最新情况 "
    "2. 如果是VIP客户，则发送感谢邮件 "
    "3. 基于搜索结果生成结构化分析报告 "
    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
"messages": [{"role": "user", "content": "请分析客户张三"}]
# "messages": [{"role": "user","content": "请分析客户李四"}]
# "messages": [{"role": "user","content": "请分析客户王五"}]
# "messages": [{"role": "user","content": "今天天气如何"}]
})
# 处理结果
rprint(result)
if "structured_response" in result:
    analysis = result["structured_response"]
    print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='fc18f529-75b2-43f9-8b4e-7187af693988'
        ),
        AIMessage(
            content='好的，我先来搜索客户张三的信息',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 
'好的，我需要按照用户的要求来分析客户张三的情况。首先，我需要搜索客户数据库了解最新情况。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 600,
                    'total_tokens': 677,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 21,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'dashscope',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-3d16f900-ff15-9681-881b-9712af43395e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f1d3b-9c89-7823-8bd5-97babf1b4937-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_8bd6b2effd384fa49387fc7f',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 600,
                'output_tokens': 77,
                'total_tokens': 677,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 21}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='4178d0fe-0a33-47d8-be03-af66d273a9b2',
            tool_call_id='call_8bd6b2effd384fa49387fc7f'
        ),
        AIMessage(
            content='找到了张三的信息！他是 **VIP客户**。现在按流程，我先发送感谢邮件，同时生成分析报告。\n\n',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '找到了张三的信息，他是VIP客户。根据要求：\n\n1. ✅ 搜索客户数据库了解最新情况
- 已完成\n2. 如果是VIP客户，则发送感谢邮件 - 张三确实是VIP客户，所以需要发送感谢邮件\n3. 
基于搜索结果生成结构化分析报告 - 需要生成报告\n4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象 - 
找到了客户信息，所以不需要返回空对象\n\n所以接下来我需要：\n1. 发送感谢邮件给张三\n2. 
生成结构化分析报告\n\n先发送邮件，然后生成报告。这两步没有依赖关系，可以同时进行。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 315,
                    'prompt_tokens': 693,
                    'total_tokens': 1008,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 134,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512}
                },
                'model_provider': 'dashscope',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-0bb859f4-0c6c-9577-acb2-7a2bc868002a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f1d3b-a3d0-7a11-b1a6-6bf6c2271675-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_116a20bbf0714af295d79a09',
                    'type': 'tool_call'
                },
                {
                    'name': 'CustomerAnalysis',
           

customer_name='张三' customer_tier='VIP客户' recent_activity='最近购买日期：2026-01-15' spending_level='高' send_email=True


### 输出模式2：TypedDict类型
TypedDict 允许为字典对象定义固定的键名和对应的 值类型 ，是 带有类型提示的字典结构 。具体的：
1. TypedDict字段定义采用 Annotated[类型, 默认值, "描述"]格式
2. 可选字段使用 Optional包装，默认值在 Annotated 中指定
3. TypedDict 不支持运行时验证。

In [6]:
from typing import TypedDict, Annotated

class ContactInfo(TypedDict):
    """用户的联系方式"""
    name: Annotated[str, ..., "用户姓名"]
    email: Annotated[str, ..., "用户邮箱地址"]
    phone: Annotated[str, ..., "用户的手机号"]
agent = create_agent(
model=model,
response_format=ToolStrategy(ContactInfo)
)
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
    ]
})
for msg in response["messages"]:
    msg.pretty_print()
rprint(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_8e38d13dbbdd40faac064ea3)
 Call ID: call_8e38d13dbbdd40faac064ea3
  Args:
    name: 小明
    email: songhk@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'}


{'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'}

JsonSchema和@dataclass类型就不列举了，差不多的

### 多schema联合模式
ToolStrategy允许指定多个类型`Union[类型1, 类型2]`这种写法，LLM能够根据输入文本的内容，智能
地选择 最合适的一个 `数据模型（Schema`）来生成结构化输出，但是最终会只有一种类型输出

In [8]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventInfo] # 联合类型
    )
)

response = agent.invoke({"messages": [
    HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")]
})
for msg in response["messages"]:
    msg.pretty_print()
rprint(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_800bf6c2ac3048e19fda666c)
 Call ID: call_800bf6c2ac3048e19fda666c
  Args:
    name: 小明
    email: shkstart@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'


ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')

## 自定义工具消息：tool_message_content参数

可以通过ToolStrategy的 tool_message_content 参数定制其消息内容，将指定的内容写入对话历
史的提示信息，这样做的好处如下：
1. 在最终用户可见的对话流中，使用 更自然的消息 替代原始数据。
2. 用简短的确认信息替代可能很长的数据块， 减少token消耗 。

In [12]:
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo,
    tool_message_content="从这段话中抽取结构化信息："),
    
)
response = agent.invoke({
"messages": [
    HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")]
})
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='5d059936-6047-4df6-b489-dceefeddea59'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 
'用户要求从一段话中提取结构化信息，包括姓名、邮箱和手机号。文本是："小明的邮箱地址为：songhk@atguigu.com，手机号：1
2345678912"\n\n很明显：\n- 姓名：小明\n- 邮箱：songhk@atguigu.com\n- 
手机号：12345678912\n\n现在调用ContactInfo工具来提交这些信息。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 169,
                    'prompt_tokens': 344,
                    'total_tokens': 513,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 83,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'dashscope',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-635e6259-1b1d-9098-a27b-92114f5c3a22',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f1e09-fca0-7ff1-93db-0630a61b118a-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_948bf3bf06be4f2391342ea1',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 344,
                'output_tokens': 169,
                'total_tokens': 513,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 83}
            }
        ),
        ToolMessage(
            content='从这段话中抽取结构化信息：',
            name='ContactInfo',
            id='70457bd0-6145-4330-9c0b-0a60e7c6247b',
            tool_call_id='call_948bf3bf06be4f2391342ea1'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

## 错误处理：handle_errors参数
ToolStrategy通过其 handle_errors参数 提供了结构化过程错误处理策略，以下是主要的几种方式及其用途：

- handle_errors=True： LangChain默认方式 ， 捕获所有异常 ，并使用LangChain 内置的、信息明确的 错误消息模板 提示模型重试，确保最终能得到符合预定格式的有效数据。适用于大多数希望自动处理错误的通用场景
。

- handle_errors=False：关闭自动重试机制，任何异常都会 直接抛出 ，会 中断程序 运行。
 
- handle_errors="自定义字符串"：捕获所有异常，但使用开发者 预设的固定字符串 作为错误消
息。适用于需要统一、友好的用户提示，或进行特定业务引导的场景。

- handle_errors=ExceptionType：仅 捕获指定类型(如ValueError) 或元组中的异常类型并进行重
试， 其他异常直接抛出 。适用于需要 精准控制 ，只对特定错误进行重试的场景。

- handle_errors=callable：灵活性最高的方式，使用开发者 自定义的函数来处理异常 ，可根据不同的异常类型返回差异化的提示信息。适用于需要复杂、精细化错误处理的场景。

In [20]:
class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")
class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(Union[ContactInfo, EventDetails],tool_message_content="提取完成！",
    # handle_errors=True
    handle_errors="请检查输入数据"
))
result = agent.invoke({"messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})
rprint(result)

KeyboardInterrupt: 

### 设置为指定异常类型
1. MultipleStructuredOutputsError：多结构化输出错误，当返回的工具调用请求数量大于1时，抛出该异常，默认情况下LangChain会拦截异常并提醒模型重试。
2. StructuredOutputValidationError：输出结构化验证错误，当输出格式不符合结构化要求时，抛出上述异常。
默认情况下LangChain会拦截该异常并自动重试。

### 设置为自定义错误处理函数
指定异常处理函数并返回字符串时，LangChain会在遇到异常时自动重试并将异常处理函数的返回值作为ToolMessage的内容。

In [21]:
from langchain.agents.structured_output import ToolStrategy,StructuredOutputValidationError, MultipleStructuredOutputsError
from rich import print as rprint

# 自定义错误处理函数
def custom_error_handler(error: Exception) -> str:
    """自定义错误处理器"""
    error_str = str(error)
    print(f"捕获到错误类型：{type(error).__name__}")
    print(f"错误详情：{error_str}")
    if isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求。"
    elif isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回。"
    else:
        return f"Error: {error_str}"

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(Union[ContactInfo, EventDetails],
    tool_message_content="提取完成！",
    handle_errors=custom_error_handler
))
result = agent.invoke({"messages": [{"role": "user","content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"}]
})
rprint(result)

KeyboardInterrupt: 